In [1]:
import pandas as pd
import numpy as np
from functions import clean_data, split_data


In [2]:
X, Y = clean_data("claims_train.csv")
X_train, X_val, y_train, y_val = split_data(X, Y, 0.2)


In [18]:
from sklearn.model_selection import train_test_split

subset_size = 5000
subset_ratio = subset_size / len(X_train)

# 1) Create a representative subset
X_subset, _, y_subset, _ = train_test_split(
    X_train, y_train,
    train_size=subset_ratio,
    stratify=y_train,   # <-- keeps class distribution
    random_state=42
)

# 2) Now split this subset into train/val for hyperparameter tuning
X_train_sub, X_val_sub, y_train_sub, y_val_sub = train_test_split(
    X_subset, y_subset,
    test_size=0.2,
    stratify=y_subset,
    random_state=42
)


ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

In [7]:
def initialize_parameters(input_layer, hidden_layer1_neurons, hidden_layer2_neurons, output_layer):
    np.random.seed(42)
    parameters = {

    "w1": np.random.randn(input_layer, hidden_layer1_neurons) * np.sqrt(2.0 / input_layer),
    "b1": np.zeros((1, hidden_layer1_neurons)),

    "w2":np.random.randn(hidden_layer1_neurons, hidden_layer2_neurons) * np.sqrt(2.0 / hidden_layer1_neurons),
    "b2": np.zeros((1, hidden_layer2_neurons)), 

    "w3": np.random.randn(hidden_layer2_neurons,output_layer) * np.sqrt(2.0 / hidden_layer2_neurons),
    "b3": np.zeros((1, output_layer))}
    return parameters

In [8]:
def relu(z):
    return np.maximum(0, z)

def softplus(z):
    return np.log1p(np.exp(z))

def relu_derivative(z):
    return (z > 0).astype(float)

def softplus_derivative(z):
    return 1 / (1 + np.exp(-z))

In [9]:
def feed_forward(X, parameters):
    w1, b1 = parameters["w1"], parameters["b1"]
    w2, b2 = parameters["w2"], parameters["b2"]
    w3, b3 = parameters["w3"], parameters["b3"]
    
    z1 = X @ w1 + b1
    a1 = relu(z1)
    
    z2 = a1 @ w2 + b2
    a2 = relu(z2)
    
    z3 = a2 @ w3 + b3
    y_pred = softplus(z3)
    
    values = {"z1": z1, "a1": a1, "z2": z2, "a2": a2, "z3": z3, "predicted_y": y_pred}
    return y_pred, values
    

In [16]:
def backward_propagation(X, y, parameters, values):
    w1, b1 = parameters["w1"], parameters["b1"]
    w2, b2 = parameters["w2"], parameters["b2"]
    w3, b3 = parameters["w3"], parameters["b3"]
    
    z1, a1 = values["z1"], values["a1"]
    z2, a2 = values["z2"], values["a2"]
    z3 = values["z3"]
    y_pred = values["predicted_y"]
    m = X.shape[0]
    
    y = y.reshape(-1, 1)
    dL_dlambda = 1 - y / (y_pred + 1e-8)
    dz3 = dL_dlambda * softplus_derivative(z3)    
    dw3 = (a2.T @ dz3) / m                       
    db3 = np.sum(dz3, axis=0, keepdims=True) / m  

    da2 = dz3 @ w3.T                              
    dz2 = da2 * relu_derivative(z2)               
    dw2 = (a1.T @ dz2) / m                       
    db2 = np.sum(dz2, axis=0, keepdims=True) / m  
    
    da1 = dz2 @ w2.T                              
    dz1 = da1 * relu_derivative(z1)              
    dw1 = (X.T @ dz1) / m                         
    db1 = np.sum(dz1, axis=0, keepdims=True) / m 
    
    gradients = {
        "w1": dw1,
        "b1": db1,
        "w2": dw2,
        "b2": db2,
        "w3": dw3,
        "b3": db3,
    }
    return gradients

In [11]:
def init_adam_state(parameters):
    adam_state = {
        "t": 0,
        "m": {},
        "v": {}
    }
    for name, value in parameters.items():
        adam_state["m"][name] = np.zeros_like(value)
        adam_state["v"][name] = np.zeros_like(value)
        
    return adam_state


In [12]:
def adam_update(parameters, grads, adam_state, learning_rate=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
    adam_state["t"] += 1
    t = adam_state["t"]

    for name in parameters.keys():
        g = grads[name]

        adam_state["m"][name] = beta1 * adam_state["m"][name] + (1 - beta1) * g
        adam_state["v"][name] = beta2 * adam_state["v"][name] + (1 - beta2) * (g ** 2)

        m_hat = adam_state["m"][name] / (1 - beta1 ** t)
        v_hat = adam_state["v"][name] / (1 - beta2 ** t)

        parameters[name] -= learning_rate * m_hat / (np.sqrt(v_hat) + eps)

    return parameters, adam_state


In [ ]:
batch_sizes = [128, 256, 512]

In [ ]:
def train_model(X_train, y_train, X_val, y_val, learning_rate, batch_size, epochs):
    parameters = initialize_parameters(
        input_layer = X_train.shape[1],
        hidden_layer1_neurons = 28,
        hidden_layer2_neurons = 28,
        output_layer = 1
    )
    
    adam_state = init_adam_state(parameters)
    best_val_loss = float('inf')
    patience_counter = 0
    best_parameters = {k: v.copy() for k, v in parameters.items()}

    train_losses = []
    val_losses = []
    
    for epoch in range(epochs):
        indices = np.arange(len(X_train))
        np.random.shuffle(indices)
        X_shuffled = X_train[indices]
        y_shuffled = y_train[indices]
        batch_losses = []
        
        for i in range(0, len(X_shuffled), batch_size):
            X_batch = X_shuffled[i:i+batch_size]
            y_batch = y_shuffled[i:i+batch_size]
            y_batch = y_batch.reshape(-1, 1) 
            
            predicted_y, values = feed_forward(X_batch, parameters)
            
            train_loss = np.mean(
            predicted_y - y_batch * np.log(predicted_y + 1e-8)
            )

            batch_losses.append(train_loss)

            gradients = backward_propagation(
                X_batch, y_batch, parameters, values,
            )

            parameters, adam_state = adam_update(
                parameters, gradients, adam_state,
                learning_rate=learning_rate  
            )
            
        epoch_train_loss = np.mean(batch_losses)
        train_losses.append(epoch_train_loss)
        
        # validation loss
        y_val = y_val.reshape(-1, 1)
        y_val_pred, _ = feed_forward(X_val, parameters)
        val_loss = np.mean(
        y_val_pred - y_val * np.log(y_val_pred + 1e-8)
        )
        
        val_losses.append(val_loss)
        
        val_mse = np.mean((y_val - y_val_pred)**2)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_parameters = {k: v.copy() for k, v in parameters.items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= 5:   
            print(f"Early stopping at epoch {epoch}")
            break
        
        print(f"Epoch {epoch}: train={epoch_train_loss:.4f}, val={val_loss:.4f}")
        
    return best_parameters, train_losses, val_losses, val_mse


In [ ]:
best_config = None
best_loss = float("inf")
epochs = 100

for batch_size in batch_sizes:

        print(f"Testing batch={batch_size}, epochs={epochs}")

        best_params, train_losses, val_losses = train_model(
            X_train_sub, y_train_sub,
            X_val_sub,   y_val_sub,
            learning_rate=0.001,
            batch_size=batch_size,
            epochs=epochs,
        )

        val_loss = min(val_losses)
        print("Validation loss:", val_loss)

        if val_loss < best_loss:
            best_loss = val_loss
            best_config = (batch_size, epochs)

print("\nBest hyperparameters:")
print(best_config)
print("Best validation loss:", best_loss)


Testing batch=128, epochs=50
Epoch 0: train=0.6556, val=0.3292
Epoch 1: train=0.2702, val=0.2276
Epoch 2: train=0.2274, val=0.2146
Epoch 3: train=0.2199, val=0.2099
Epoch 4: train=0.2236, val=0.2071
Epoch 5: train=0.2146, val=0.2051
Epoch 6: train=0.2117, val=0.2036
Epoch 7: train=0.2133, val=0.2025
Epoch 8: train=0.2122, val=0.2020
Epoch 9: train=0.2137, val=0.2017
Epoch 10: train=0.2174, val=0.2011
Epoch 11: train=0.2196, val=0.2013
Epoch 12: train=0.2138, val=0.2016
Epoch 13: train=0.2146, val=0.2017
Epoch 14: train=0.2140, val=0.2017
Early stopping at epoch 15
Validation loss: 0.20109496515682546
Testing batch=128, epochs=70
Epoch 0: train=0.6556, val=0.3292
Epoch 1: train=0.2702, val=0.2276
Epoch 2: train=0.2274, val=0.2146
Epoch 3: train=0.2199, val=0.2099
Epoch 4: train=0.2236, val=0.2071
Epoch 5: train=0.2146, val=0.2051
Epoch 6: train=0.2117, val=0.2036
Epoch 7: train=0.2133, val=0.2025
Epoch 8: train=0.2122, val=0.2020
Epoch 9: train=0.2137, val=0.2017
Epoch 10: train=0.2174,